# 04 - 检查知识图谱

检查共表达图和 GO 相似度图的结构、统计信息和可视化。

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path

# 检查共表达图
coexp_dir = Path('../graphs/coexpression')
if (coexp_dir / 'edges.csv').exists():
    edges = pd.read_csv(coexp_dir / 'edges.csv')
    nodes = pd.read_csv(coexp_dir / 'genes.csv')
    print(f'Co-expression graph:')
    print(f'  Nodes: {len(nodes)}')
    print(f'  Edges: {len(edges)}')
    print(f'  Avg degree: {2*len(edges)/len(nodes):.2f}')
    print(f'  Weight range: [{edges["weight"].min():.3f}, {edges["weight"].max():.3f}]')
else:
    print('Co-expression graph not built yet.')
    print('Run: python scripts/build_coexpression_graph.py --adata <path>')

In [ ]:
# 检查 GO 相似度图
go_dir = Path('../graphs/go_similarity')
if (go_dir / 'edges.csv').exists():
    go_edges = pd.read_csv(go_dir / 'edges.csv')
    go_nodes = pd.read_csv(go_dir / 'genes.csv')
    print(f'GO similarity graph:')
    print(f'  Nodes: {len(go_nodes)}')
    print(f'  Edges: {len(go_edges)}')
else:
    print('GO similarity graph not built yet.')
    print('Run: python scripts/build_go_graph.py --adata <path>')
    print('Note: requires GAF annotation file for full construction.')

In [ ]:
# 图统计和可视化 (如果图已构建)
if (coexp_dir / 'edges.csv').exists():
    G = nx.from_pandas_edgelist(edges, 'source', 'target', edge_attr='weight')
    print(f'Graph stats:')
    print(f'  Connected components: {nx.number_connected_components(G)}')
    print(f'  Density: {nx.density(G):.4f}')
    
    # 度分布
    degrees = [d for _, d in G.degree()]
    print(f'  Degree: mean={np.mean(degrees):.1f}, max={max(degrees)}, min={min(degrees)}')
    
    # top 连接基因
    gene_map = dict(zip(nodes['node_id'], nodes['gene_name']))
    top_connected = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]
    print()
    print('Top 10 most connected genes:')
    for node_id, deg in top_connected:
        print(f'  {gene_map.get(node_id, node_id)}: {deg} neighbors')